# Turbofan Remaining Useful Life Prediction

Using C-MAPSS FD001 sensor data to predict how many operating cycles an engine has left. We compare Linear Regression, Ridge and Random Forest.

Run the notebook from the project root or the `notebooks` folder. RUL is a regression target, so we use MAE, RMSE and R² instead of classification accuracy.


## 1. Imports

Load the libraries used in this notebook.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.base import clone
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. Load FD001

Each row contains an engine ID, a cycle number, three operating settings and 21 sensor readings.

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

train_path = PROJECT_ROOT / "data/train/train_FD001.txt"
test_path = PROJECT_ROOT / "data/test/test_FD001.txt"
rul_path = PROJECT_ROOT / "data/rul/RUL_FD001.txt"

column_names = ["unit_number", "time_cycles"]
column_names += [f"operational_setting_{i}" for i in range(1, 4)]
sensor_columns = [f"sensor_{i}" for i in range(1, 22)]
column_names += sensor_columns

# The files use spaces between values and have no header row.
df = pd.read_csv(train_path, sep=r"\s+", header=None)
assert df.shape[1] == len(column_names), "Expected 26 columns."
df.columns = column_names

print("Shape:", df.shape)
print("Training engines:", df["unit_number"].nunique())
df.head()


## 3. Check the data

Check missing values and duplicate engine/cycle records before training. Keep the sensor values as decimals.

In [ ]:
df.info()
print("Missing values:", df.isna().sum().sum())
print("Duplicate records:", df.duplicated(["unit_number", "time_cycles"]).sum())

assert not df.isna().any().any(), "Inspect missing values before continuing."
assert not df.duplicated(["unit_number", "time_cycles"]).any(), "Inspect duplicate records."

df = df.sort_values(["unit_number", "time_cycles"]).reset_index(drop=True)
df.describe().T


## 4. Explore engine life and a sensor

Compare engine lifetimes, then inspect sensor 2 for engine 1.

In [ ]:
engine_life = df.groupby("unit_number")["time_cycles"].max()
display(engine_life.describe())

plt.figure(figsize=(10, 4))
plt.plot(engine_life.index, engine_life.values)
plt.title("Training engine lifetimes")
plt.xlabel("Engine number")
plt.ylabel("Lifetime (cycles)")
plt.tight_layout()
plt.show()

engine_1 = df[df["unit_number"] == 1]
plt.figure(figsize=(10, 4))
plt.plot(engine_1["time_cycles"], engine_1["sensor_2"])
plt.title("Sensor 2 — engine 1")
plt.xlabel("Operating cycle")
plt.ylabel("Sensor reading")
plt.tight_layout()
plt.show()


## 5. Create the RUL target

Training engines run to failure. Their RUL is the last cycle minus the current cycle. This formula must not be used for the truncated test data.

In [ ]:
# Use the failure cycle to create labels, not as a model input.
last_training_cycle = df.groupby("unit_number")["time_cycles"].transform("max")
df["RUL"] = last_training_cycle - df["time_cycles"]

df.loc[df["unit_number"] == 1, ["unit_number", "time_cycles", "RUL"]].tail()


## 6. Split by engine

Use 80% of engines for training and 20% for model selection. No engine should appear in both sets.

In [ ]:
feature_candidates = [col for col in column_names if col != "unit_number"]
X = df[feature_candidates]
y = df["RUL"]
groups = df["unit_number"]

# Grouped splitting keeps readings from the same engine together.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(X, y, groups=groups))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
groups_train = groups.iloc[train_idx]

train_engines = set(groups.iloc[train_idx])
val_engines = set(groups.iloc[val_idx])
assert train_engines.isdisjoint(val_engines), "An engine appears in both sets."

print("Training:", X_train.shape, "| engines:", len(train_engines))
print("Validation:", X_val.shape, "| engines:", len(val_engines))


## 7. Train the models

VarianceThreshold removes constant columns. The linear models also need scaling; Random Forest does not. Pipelines keep these transformations attached to the model.

In [ ]:
linear_model = make_pipeline(
    VarianceThreshold(),
    StandardScaler(),
    LinearRegression()
)

ridge_model = make_pipeline(
    VarianceThreshold(),
    StandardScaler(),
    Ridge(alpha=1.0)
)

rf_model = make_pipeline(
    VarianceThreshold(),
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
)

models = {
    "Linear Regression": linear_model,
    "Ridge Regression": ridge_model,
    "Random Forest": rf_model
}

# Fit preprocessing on training rows only; pass raw inputs to each pipeline.
for name, model in models.items():
    model.fit(X_train, y_train)
    print("Trained:", name)


## 8. Compare validation results

MAE and RMSE are errors in cycles: lower is better. Higher R² is better, but it is not an accuracy percentage.

In [ ]:
def evaluate_model(name, actual, predicted):
    return {
        "Model": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted)
    }

rows = []
for name, model in models.items():
    predicted = model.predict(X_val)
    rows.append(evaluate_model(name, y_val, predicted))

results = pd.DataFrame(rows).sort_values("RMSE")
results.round(3)


## 9. Tune Ridge

Try different alpha values using five engine-wise folds within the training set. The pipeline fits its preprocessing separately inside each fold.

In [ ]:
ridge_pipeline = make_pipeline(
    VarianceThreshold(),
    StandardScaler(),
    Ridge()
)

param_grid = {"ridge__alpha": [0.001, 0.01, 0.1, 1, 10, 100]}
group_cv = GroupKFold(n_splits=5)

ridge_search = GridSearchCV(
    ridge_pipeline,
    param_grid,
    cv=group_cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=1
)
ridge_search.fit(X_train, y_train, groups=groups_train)

print("Best alpha:", ridge_search.best_params_)
print("CV RMSE:", round(-ridge_search.best_score_, 3))

models["Tuned Ridge"] = ridge_search.best_estimator_
tuned_predictions = models["Tuned Ridge"].predict(X_val)
tuned_result = evaluate_model("Tuned Ridge", y_val, tuned_predictions)

results = pd.DataFrame(rows + [tuned_result]).sort_values("RMSE")
results.round(3)


## 10. Fit the selected model

Select using validation RMSE, then refit that model on all FD001 training data. Do not change the selection after seeing official test scores.

In [ ]:
best_model_name = results.iloc[0]["Model"]
final_pipeline = clone(models[best_model_name])
final_pipeline.fit(X, y)

print("Selected model:", best_model_name)


## 11. Evaluate the official test data

Use the last observed cycle of each test engine. Sort by engine number to match the official RUL labels. These are endpoint scores, unlike validation scores across all cycles.

In [ ]:
test_df = pd.read_csv(test_path, sep=r"\s+", header=None)
assert test_df.shape[1] == len(column_names), "Test columns do not match training."
test_df.columns = column_names

last_indices = test_df.groupby("unit_number")["time_cycles"].idxmax()
test_last = test_df.loc[last_indices].sort_values("unit_number").reset_index(drop=True)
true_rul = pd.read_csv(rul_path, sep=r"\s+", header=None).iloc[:, 0]

# Each label must align with the endpoint for the same engine number.
assert len(test_last) == len(true_rul), "Test engines and RUL labels do not match."
assert np.array_equal(test_last["unit_number"], np.arange(1, len(test_last) + 1))

X_test = test_last[feature_candidates]
test_predictions = final_pipeline.predict(X_test)
test_metric_row = evaluate_model(best_model_name, true_rul, test_predictions)

final_metrics = pd.DataFrame([test_metric_row])
print("Test engines:", len(test_last))
final_metrics.round(3)


## 12. Inspect prediction errors

Positive error means the model overestimates remaining life. Check the largest errors as well as the average score.

In [ ]:
comparison = pd.DataFrame({
    "unit_number": test_last["unit_number"],
    "Actual_RUL": true_rul,
    "Predicted_RUL": test_predictions
})
comparison["Error"] = comparison["Predicted_RUL"] - comparison["Actual_RUL"]
comparison["Absolute_Error"] = comparison["Error"].abs()

display(comparison.head().round(2))
display(comparison.nlargest(10, "Absolute_Error").round(2))

plt.figure(figsize=(7, 5))
plt.scatter(comparison["Actual_RUL"], comparison["Predicted_RUL"], alpha=0.7)
limits = [min(true_rul.min(), test_predictions.min()), max(true_rul.max(), test_predictions.max())]
plt.plot(limits, limits, "r--")
plt.xlabel("Actual RUL (cycles)")
plt.ylabel("Predicted RUL (cycles)")
plt.title("Actual vs predicted RUL")
plt.tight_layout()
plt.show()


## 13. Save the model and results

The complete pipeline includes the fitted preprocessing. Export is optional and uses a separate folder so your older result files are preserved.

In [ ]:
SAVE_RESULTS = False  # Set True when you want to save this run.
results_dir = PROJECT_ROOT / "results/cleaned_fd001_run"

if SAVE_RESULTS:
    results_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(final_pipeline, results_dir / "final_model_pipeline.joblib")
    comparison.to_csv(results_dir / "fd001_test_predictions.csv", index=False)
    final_metrics.to_csv(results_dir / "final_metrics.csv", index=False)
    results.to_csv(results_dir / "validation_model_comparison.csv", index=False)
    print("Saved to:", results_dir)
else:
    print("Export is disabled.")


## Conclusion

Use the model and scores printed above when reporting this run. The validation engines selected the model; the official test endpoints measured its final benchmark performance. This does not establish performance on other FD subsets or real engines.
